# 06 — Interprétabilité : t-SNE et UMAP (Partie 6)
**Responsable : DIOP Alioune** · relecture : FALL Madjiguène

Mise au point sur le **modèle V1** (configuration retenue dans `V1_one_hidden_layer.ipynb`, lue dans
`results/logs/v1_selected.json`) et sur la **validation** : ce notebook n'accède jamais au test.
Les fonctions de `src/har/interpret.py` sont ensuite réutilisées telles quelles dans la V2.

## 6.1 Lectures : principe et limites

**t-SNE** (van der Maaten & Hinton, 2008). On transforme les distances entre points en probabilités de voisinage
(noyau gaussien dans l'espace d'origine, dont la largeur est fixée pour chaque point par la *perplexité*, ≈ nombre
effectif de voisins), puis on cherche des positions 2D dont les probabilités de voisinage (noyau de Student, à queues
lourdes) sont les plus proches au sens de la divergence de Kullback-Leibler. Il préserve surtout la **structure locale**.

**UMAP** (McInnes et al., 2018). Construit un graphe de k plus proches voisins pondéré (hypothèse d'une variété
localement uniforme), puis optimise une disposition 2D qui conserve ce graphe (entropie croisée floue). Plus rapide,
préserve souvent un peu mieux la structure globale ; paramètres clés : `n_neighbors` (local ↔ global), `min_dist`.

**Limites et précautions** :
1. les **tailles des groupes** et les **distances entre groupes** sur la figure ne sont pas interprétables
   (surtout en t-SNE) : seule la proximité locale l'est ;
2. le résultat dépend des **hyperparamètres** (perplexité, `n_neighbors`) → on en teste plusieurs ;
3. il est **stochastique** → on compare plusieurs graines ;
4. des groupes peuvent apparaître dans du bruit, ou se fondre alors qu'ils sont séparables en dimension élevée →
   on complète par des mesures calculées **dans l'espace d'origine** (silhouette, pureté des k plus proches voisins).

In [ ]:
import sys, os, json
sys.path.append("../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from har import config, data, models, training, metrics, plots, interpret

config.FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=0.9)
FAST = os.environ.get("HAR_FAST") == "1"
print("UMAP disponible :", interpret.umap_available(), "(sinon : pip install umap-learn)")

## Données et modèle V1 retenu

In [ ]:
df = data.load_train()
df_tr, df_va, val_subjects = data.split_by_subject(df)
X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]

sel_path = config.LOG_DIR / "v1_selected.json"
if sel_path.exists():
    sel = json.loads(sel_path.read_text())
else:   # si la V1 n'a pas encore été exécutée
    sel = {"hidden": [64], "activation": "relu", "optimizer": "adam", "lr": 1e-3, "epochs": 60}
    print("⚠️ v1_selected.json absent : configuration par défaut")
if FAST: sel["epochs"] = 5
print("configuration :", {k: sel[k] for k in ("hidden", "activation", "optimizer", "lr", "epochs")})

training.set_seed(0)
model = models.MLP(IN_DIM, sel["hidden"], activation=sel["activation"])
opt = training.make_optimizer(sel["optimizer"], model.parameters(), lr=sel["lr"])
hist = training.fit(model, train_loader, val_loader, opt, epochs=sel["epochs"])
ev = training.evaluate(model, val_loader, nn.CrossEntropyLoss(), return_preds=True)
cm_val = metrics.confusion(ev["y_true"], ev["y_pred"])
print(f"validation : accuracy {ev['accuracy']:.4f} · F1 macro {ev['f1_macro']:.4f}")
plots.confusion_heatmap(cm_val, "MLP V1 — validation", save="p6_confusion_val");

## 6.2 Projections : entrées brutes vs dernière couche cachée
Mêmes points (fenêtres de validation), mêmes paramètres de projection ; seule la représentation change :
les 561 variables standardisées d'un côté, les activations de la couche cachée de l'autre.

In [ ]:
H_va = interpret.hidden_activations(model, X_va)
print("dimension de la couche cachée :", H_va.shape[1])
fig, Z_in_tsne, Z_hid_tsne = interpret.compare_spaces(model, X_va, y_va, method="tsne", perplexity=30, save="p6_tsne_v1")

In [ ]:
if interpret.umap_available():
    fig, Z_in_umap, Z_hid_umap = interpret.compare_spaces(model, X_va, y_va, method="umap",
                                                           n_neighbors=15, min_dist=0.1, save="p6_umap_v1")
else:
    print("UMAP non installé : seule la t-SNE est présentée.")

### Mesures de séparation (indépendantes de la figure)
Silhouette et pureté des 10 plus proches voisins, calculées **dans l'espace d'origine** (561 dim. ou couche cachée)
et dans les projections 2D. Plus c'est haut, mieux les activités sont séparées.

In [ ]:
spaces = {"entrées brutes (561 d)": X_va, "couche cachée": H_va,
          "t-SNE entrées": Z_in_tsne, "t-SNE couche cachée": Z_hid_tsne}
if interpret.umap_available():
    spaces.update({"UMAP entrées": Z_in_umap, "UMAP couche cachée": Z_hid_umap})
sep = interpret.separation_scores(spaces, y_va, k=10).round(3)
sep

## 6.3 Analyse : chevauchements vs confusions du modèle
Pour chaque paire de classes, part des 10 plus proches voisins d'une classe qui appartiennent à l'autre
(« chevauchement »), comparée au taux de confusion du modèle sur la validation.

In [ ]:
M_hid = interpret.neighbor_mixing(H_va, y_va, k=10)
M_in = interpret.neighbor_mixing(X_va, y_va, k=10)
interpret.mixing_heatmaps(M_hid, cm_val, "chevauchement k-NN (couche cachée)", save="p6_mixing_vs_confusion");

In [ ]:
pairs = interpret.mixing_vs_confusion(M_hid, cm_val)
pairs["chevauchement k-NN (entrées)"] = [M_in[config.CLASSES.index(a), config.CLASSES.index(b)]
                                        for a, b in zip(pairs["vraie classe"], pairs["confondue avec"])]
if pairs["taux de confusion"].std() > 0 and pairs["chevauchement k-NN"].std() > 0:
    r = np.corrcoef(pairs["taux de confusion"], pairs["chevauchement k-NN"])[0, 1]
    print(f"corrélation (30 paires) entre confusion et chevauchement dans la couche cachée : {r:.2f}")
else:
    print("corrélation non calculable (aucune confusion ou aucun chevauchement)")
pairs.head(8).round(3)

### Limites : perplexité et variabilité d'une exécution à l'autre

In [ ]:
interpret.perplexity_grid(H_va, y_va, perplexities=(5, 30, 50, 100), save="p6_perplexity");

In [ ]:
interpret.seed_grid(H_va, y_va, seeds=(0, 1, 2), method="tsne", perplexity=30, save="p6_seeds");

In [ ]:
# Variabilité chiffrée : la pureté k-NN de la projection change-t-elle avec la graine ?
pur = [np.diag(interpret.neighbor_mixing(interpret.project(H_va, "tsne", s, perplexity=30), y_va)).mean()
       for s in (0, 1, 2)]
print("pureté 10-NN de la t-SNE (couche cachée), 3 graines :", metrics.mean_std(pur))

## Analyse (à rédiger à partir des sorties)

- **Ce que le réseau a changé** : dans les entrées brutes, on distingue en général deux grands ensembles (activités
  dynamiques : marche, montée, descente / activités statiques : assis, debout, allongé), avec LAYING bien isolé ;
  dans la couche cachée, les six activités forment-elles des groupes plus compacts ? Appuyer avec la silhouette et la
  pureté k-NN (tableau « mesures de séparation »).
- **Lien avec la matrice de confusion** : les paires qui se chevauchent (SITTING ↔ STANDING attendu, éventuellement
  UPSTAIRS ↔ DOWNSTAIRS) sont-elles celles que le modèle confond ? (tableau des paires, corrélation).
  Explication capteurs : assis et debout sont deux postures immobiles ; l'accéléromètre à la ceinture mesure surtout
  la gravité, dont l'orientation diffère peu entre les deux.
- **Capacité à séparer les activités** : ce que la couche cachée apprend en plus d'une séparation linéaire des
  entrées (rappel : la régression logistique fait déjà très bien).
- **Précautions** : ne pas lire les distances entre groupes ni leurs tailles ; les figures varient avec la perplexité
  et la graine (figures ci-dessus) ; les conclusions s'appuient sur les mesures en dimension d'origine.